# GeoMapComp — Distributed Raster Search

Cleaned, path-configurable companion notebook for the paper
**"Scalable Approximate Computing for Efficient Search in Satellite Remote
Sensing Products Using Apache Spark"** (IDSS 2025).

This notebook mirrors `main.py`. Set the two paths below (no Google Colab or
Drive required) and run all cells.

In [ ]:
# Install dependencies if needed (uncomment)
# !pip install pyspark numpy pandas scipy pygeohash GDAL

In [ ]:
import os
import sys

# Make the src package importable when running from the repo root.
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))

from pyspark.sql import SparkSession
from src.geomapcomp import run_search

## Configuration

Edit these two paths to point at your data, or set the
`GEOMAPCOMP_SAMPLES_PATH` / `GEOMAPCOMP_REFERENCE_PATH` environment variables.

In [ ]:
SAMPLES_PATH = os.environ.get("GEOMAPCOMP_SAMPLES_PATH", "../data/samples")
REFERENCE_PATH = os.environ.get("GEOMAPCOMP_REFERENCE_PATH", "../data/AQ_NYC_Reference.tiff")
NUM_WORKERS = 3
PRECISION = 7

## Start Spark

In [ ]:
spark = (
    SparkSession.builder
    .appName("GeoMapComp Distributed Raster Search")
    .master(f"local[{NUM_WORKERS}]")
    .config("spark.executor.memory", "2g")
    .getOrCreate()
)

## Run the search for each metric

For every metric, lower is better, so the search minimizes.

In [ ]:
metrics = [
    "RMSE",
    "MAPE",
    "Jensen_Shannon_Divergence",
    "Kullback_Leibler_Divergence",
]

results = {}
for metric in metrics:
    result = run_search(
        spark,
        samples_path=SAMPLES_PATH,
        reference_path=REFERENCE_PATH,
        metric=metric,
        num_workers=NUM_WORKERS,
        precision=PRECISION,
    )
    results[metric] = result
    print(f"{metric}: best = {result['best_sample']} "
          f"({result['metric_value']}) in {result['execution_time']:.2f}s")
    print("  per-worker:", result["per_worker"])

## Summary

In [ ]:
import pandas as pd

summary = pd.DataFrame(
    [
        {
            "metric": m,
            "best_sample": r["best_sample"],
            "value": r["metric_value"],
            "seconds": round(r["execution_time"], 2),
        }
        for m, r in results.items()
    ]
)
summary

In [ ]:
spark.stop()